# ADVC — Local Jupyter Notebook (UoM H100 Workstation)

**Environment:** 10 GB VRAM H100 slice · 1 core Intel Xeon Platinum 8480+ · 28 GB RAM

Run cells top-to-bottom in order. Each phase is resumable — re-running a cell skips already-completed rows.

**Config changes already applied in `configs/base.yaml`:**
- `eval.batch_size`: 64 → 32 (8 GB VRAM)
- `defense.batch_size`: 32 → 16 (8 GB VRAM)
- `eval.num_workers`: 2 → 0 (1 CPU core — avoids DataLoader deadlock)

In [ ]:
# Cell 0 — Set working directory to ADVC repo root (run this first, always)
import os
from pathlib import Path

_cwd = Path().resolve()

# Check common locations in order
_candidates = [
    Path('/root/data'),            # UoM workstation location
    Path('/root/data/ADVC'),
    _cwd,
    _cwd.parent,
    _cwd.parent.parent,
    Path.home() / 'ADVC',
]

for candidate in _candidates:
    if (candidate / 'configs' / 'base.yaml').exists():
        os.chdir(str(candidate))
        print('Working directory set to: ' + str(candidate))
        break
else:
    print('WARNING: Could not find repo root automatically.')
    print('Current cwd: ' + str(_cwd))
    print('Manually run:  os.chdir("/root/data")  then re-run this cell.')

print('cwd is now: ' + os.getcwd())

## Cell 1 — Verify GPU

Confirm H100 is available. The check no longer enforces A100 — any H100 is fine.

In [ ]:
# Cell 1 — Verify GPU
import torch

print('CUDA available : ' + str(torch.cuda.is_available()))
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    free_vram, total_vram = torch.cuda.mem_get_info(0)
    print('GPU name       : ' + gpu_name)
    print(f'Free VRAM      : {free_vram/1e9:.1f} GB  /  {total_vram/1e9:.1f} GB total')
    if 'H100' not in gpu_name:
        print()
        print('WARNING: Expected H100 but got: ' + gpu_name)
        print('Verify your runtime is assigned to the H100 slice.')
    else:
        print()
        print('H100 confirmed. Ready to proceed.')
else:
    print()
    print('WARNING: No CUDA GPU detected. Check your JupyterHub resource allocation.')

## Cell 2 — Install Dependencies

Install required packages. On the UoM workstation these may already be installed — the `-q` flag suppresses noise and the cell is idempotent.

In [ ]:
# Cell 2 — Install Dependencies
import subprocess, sys

packages = ['timm', 'torchattacks', 'bitsandbytes', 'optimum', 'pyyaml', 'tqdm']

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q'] + packages,
    capture_output=True, text=True
)
if result.returncode != 0:
    print('pip error:\n' + result.stderr)
else:
    print('All packages ready: ' + ', '.join(packages))

## Cell 3 — Extract Imagenette zip and Set Paths

**Edit `ZIP_PATH` to point to your uploaded imagenette zip file.**

The zip will be extracted into `data/imagenette/` inside the project root.
Label remapping (imagenette 10-class → ImageNet-1k indices) is already handled by the experiment scripts.

In [ ]:
# Cell 3 — Extract Imagenette zip and Set Paths
import os, sys, yaml, zipfile
from pathlib import Path

# ── EDIT THIS LINE — full path to the zip you uploaded ────────────────────────
ZIP_PATH = '/path/to/imagenette2-320.zip'   # e.g. '/home/user/imagenette2-320.zip'
# ─────────────────────────────────────────────────────────────────────────────

# Resolve PROJECT_ROOT as the directory containing this notebook's parent.
# __file__ is not available in notebooks, so we use the known relative location:
# this notebook lives at ADVC/notebooks/, so PROJECT_ROOT is ADVC/.
_THIS_NOTEBOOK = Path(os.path.abspath('notebooks/run_phases_local.ipynb'))
PROJECT_ROOT   = str(_THIS_NOTEBOOK.parent.parent)

# Fallback: if the notebook is opened from a different cwd, walk up from cwd
# until we find configs/base.yaml
if not os.path.exists(os.path.join(PROJECT_ROOT, 'configs', 'base.yaml')):
    _search = Path(os.getcwd())
    for candidate in [_search, _search.parent, _search.parent.parent]:
        if (candidate / 'configs' / 'base.yaml').exists():
            PROJECT_ROOT = str(candidate)
            break

print('Project root : ' + PROJECT_ROOT)
assert os.path.exists(os.path.join(PROJECT_ROOT, 'configs', 'base.yaml')), \
    'Cannot find configs/base.yaml — make sure you cloned the repo and opened the notebook from inside it.'

EXTRACT_DIR = os.path.join(PROJECT_ROOT, 'data')

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print('Working dir  : ' + os.getcwd())

# ── Extract ───────────────────────────────────────────────────────────────────
if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        f'Zip not found: {ZIP_PATH}\n'
        'Upload your imagenette zip via the Jupyter file browser, then update ZIP_PATH above.'
    )

print(f'Extracting {ZIP_PATH} ...')
os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(EXTRACT_DIR)
print('Extraction complete.')

# ── Find the extracted root (handles imagenette2-320/, imagenette2/, etc.) ────
candidates = [
    d for d in os.listdir(EXTRACT_DIR)
    if d.startswith('imagenette') and os.path.isdir(os.path.join(EXTRACT_DIR, d))
]
if not candidates:
    raise RuntimeError('Could not find an imagenette* folder inside ' + EXTRACT_DIR)
IMAGENETTE_ROOT = os.path.join(EXTRACT_DIR, candidates[0])
print('Imagenette root: ' + IMAGENETTE_ROOT)

# ── Write paths into configs/base.yaml ────────────────────────────────────────
CONFIG_PATH = os.path.join(PROJECT_ROOT, 'configs', 'base.yaml')
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

cfg['dataset']['val_dir']   = os.path.join(IMAGENETTE_ROOT, 'val')
cfg['dataset']['train_dir'] = os.path.join(IMAGENETTE_ROOT, 'train')

with open(CONFIG_PATH, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

print()
print('configs/base.yaml updated:')
print('  val_dir  : ' + cfg['dataset']['val_dir'])
print('  train_dir: ' + cfg['dataset']['train_dir'])

# ── Verify ────────────────────────────────────────────────────────────────────
ok = True
for label, path in [('val', cfg['dataset']['val_dir']), ('train', cfg['dataset']['train_dir'])]:
    if os.path.isdir(path):
        n = len(os.listdir(path))
        print(f'  OK   {label}/ found  ({n} class folders)')
    else:
        print(f'  MISSING: {path}')
        ok = False

if ok:
    print()
    print('Imagenette ready. Proceed to Cell 4.')
else:
    print()
    print('ERROR: extraction may have failed — check the zip contents and re-run.')

## Cell 4 — Create Output Directories

In [ ]:
# Cell 4 — Create Output Directories
import os

dirs = [
    'results',
    'results/checkpoints/at',
    'results/checkpoints/atkd',
    'results/figures',
]

for d in dirs:
    os.makedirs(d, exist_ok=True)
    print('Ready: ' + d)

print()
print('All directories ready.')

## Cell 5 — Smoke Test: Load Model

Load FP32 DeiT-S and run one forward pass to confirm the environment is working before starting long experiments.

> **bitsandbytes on Linux:** INT8/INT4 require `bitsandbytes>=0.41` with CUDA support. If `load_model` fails for int8/int4, confirm with `python -m bitsandbytes` that CUDA kernels loaded.

In [ ]:
# Cell 5 — Smoke Test: Load FP32 DeiT-S
import torch
from models.loader import load_config, load_model

cfg = load_config('configs/base.yaml')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Loading FP32 DeiT-S ...')
model = load_model('deit_small', 'fp32', cfg, device=device)
model.eval()

dummy = torch.randn(2, 3, 224, 224).to(device)
with torch.no_grad():
    out = model(dummy)
    # Handle HuggingFace dataclass output
    if hasattr(out, 'logits'):
        out = out.logits

print(f'Output shape: {out.shape}  (expected [2, 1000])')
print(f'VRAM used   : {torch.cuda.memory_allocated()/1e6:.0f} MB')

del model, dummy
torch.cuda.empty_cache()
print()
print('Smoke test passed. Environment is ready.')

## Cell 6 — Run Phase 1: No Defense (Baseline)

Sweeps FP32/INT8/INT4 × FGSM/PGD/Patch. Writes 9 rows to `results/phase1_results.csv`.

**Estimated time on H100 8GB slice:** ~45–60 min (patch attack dominates).

The cell is fully resumable — kill and re-run at any point.

In [ ]:
# Cell 6 — Run Phase 1: No Defense
import subprocess, sys, os

SCRIPT = os.path.join('experiments', 'eval_phase1.py')
RESULTS_FILE = os.path.join('results', 'phase1_results.csv')

print('=' * 60)
print('Phase 1: No-defense baseline sweep')
print('=' * 60)

proc = subprocess.Popen(
    [sys.executable, SCRIPT, '--model', 'deit_small'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
print()
print('[Phase 1] Exited with code ' + str(proc.returncode))
if os.path.exists(RESULTS_FILE):
    print('[Phase 1] Results saved to ' + RESULTS_FILE)
else:
    print('[Phase 1] WARNING: results file not found — check output above for errors.')

## Cell 7 — Run Phase 2a: Adversarial Training (AT) Defense

Fine-tunes compressed DeiT-S with AT (7 epochs each), then evaluates all 3 attacks.
Writes 9 rows to `results/phase2_at_results.csv`.

**Estimated time on H100 8GB slice:** ~2.5–3 hr (7 epochs × 3 compression levels × training + eval).

Checkpoints saved after every epoch to `results/checkpoints/at/`.

In [ ]:
# Cell 7 — Run Phase 2a: AT Defense
import subprocess, sys, os

SCRIPT = os.path.join('experiments', 'eval_phase2_at.py')
RESULTS_FILE = os.path.join('results', 'phase2_at_results.csv')

print('=' * 60)
print('Phase 2a: AT defense sweep')
print('=' * 60)

proc = subprocess.Popen(
    [sys.executable, SCRIPT],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
print()
print('[Phase 2a] Exited with code ' + str(proc.returncode))
if os.path.exists(RESULTS_FILE):
    print('[Phase 2a] Results saved to ' + RESULTS_FILE)
else:
    print('[Phase 2a] WARNING: results file not found — check output above for errors.')

## Cell 7b — (Optional) Run a Single Compression Level for Phase 2a

Use this if you want to run one level at a time (e.g. to check results before continuing).
Set `COMPRESSION` to `fp32`, `int8`, or `int4`.

In [ ]:
# Cell 7b — Phase 2a: Single Compression Level
import subprocess, sys, os

COMPRESSION = 'fp32'   # change to 'int8' or 'int4'
SCRIPT = os.path.join('experiments', 'eval_phase2_at.py')

print(f'Running Phase 2a for compression={COMPRESSION} only ...')

proc = subprocess.Popen(
    [sys.executable, SCRIPT, '--compression', COMPRESSION],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
print('[Phase 2a single] Exited with code ' + str(proc.returncode))

## Cell 8 — Run Phase 2b: AT+KD Defense

Same structure as Phase 2a but adds a frozen FP32 teacher for KL-divergence supervision.
Both teacher and student are loaded simultaneously — peak VRAM ~2.5 GB (fits in 8 GB slice).

Writes 9 rows to `results/phase2_atkd_results.csv`. Checkpoints in `results/checkpoints/atkd/`.

In [ ]:
# Cell 8 — Run Phase 2b: AT+KD Defense
import subprocess, sys, os

SCRIPT = os.path.join('experiments', 'eval_phase2_atkd.py')
RESULTS_FILE = os.path.join('results', 'phase2_atkd_results.csv')

print('=' * 60)
print('Phase 2b: AT+KD defense sweep')
print('=' * 60)

proc = subprocess.Popen(
    [sys.executable, SCRIPT],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
print()
print('[Phase 2b] Exited with code ' + str(proc.returncode))
if os.path.exists(RESULTS_FILE):
    print('[Phase 2b] Results saved to ' + RESULTS_FILE)
else:
    print('[Phase 2b] WARNING: results file not found — check output above for errors.')

## Cell 9 — Run Phase 3: Combined Attack

Requires `attacks/combined.py` and `experiments/eval_phase3.py` to be built first (see CLAUDE.md build order).
Writes 9 rows to `results/phase3_results.csv`.

In [ ]:
# Cell 9 — Run Phase 3: Combined Attack
import subprocess, sys, os

SCRIPT = os.path.join('experiments', 'eval_phase3.py')
RESULTS_FILE = os.path.join('results', 'phase3_results.csv')

if not os.path.exists(SCRIPT):
    print('ERROR: ' + SCRIPT + ' does not exist yet.')
    print('Build attacks/combined.py and experiments/eval_phase3.py first (see CLAUDE.md).')
else:
    print('=' * 60)
    print('Phase 3: Combined attack sweep')
    print('=' * 60)

    proc = subprocess.Popen(
        [sys.executable, SCRIPT],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in proc.stdout:
        print(line, end='', flush=True)

    proc.wait()
    print()
    print('[Phase 3] Exited with code ' + str(proc.returncode))
    if os.path.exists(RESULTS_FILE):
        print('[Phase 3] Results saved to ' + RESULTS_FILE)
    else:
        print('[Phase 3] WARNING: results file not found — check output above.')

## Cell 10 — Preview All Results

Load all completed CSVs and display as formatted tables. Run at any point to see progress.

In [ ]:
# Cell 10 — Preview All Results
import pandas as pd
import os

report = {
    'Phase 1 -- No Defense':      'results/phase1_results.csv',
    'Phase 2a -- AT Defense':     'results/phase2_at_results.csv',
    'Phase 2b -- AT+KD Defense':  'results/phase2_atkd_results.csv',
    'Phase 3 -- Combined Attack': 'results/phase3_results.csv',
}

display_cols = ['compression', 'defense', 'attack', 'clean_acc', 'robust_acc', 'asr', 'robustness_gap']

for title, path in report.items():
    print()
    print('=' * 60)
    print(title)
    print('=' * 60)
    if not os.path.exists(path):
        print('  Not yet generated: ' + path)
        continue
    df = pd.read_csv(path)
    if df.empty:
        print('  File exists but has no data rows yet.')
        continue
    cols = [c for c in display_cols if c in df.columns]
    print(df[cols].to_string(index=False))
    print()
    print(f'  {len(df)} row(s) in {path}')

## Cell 11 — VRAM Monitor

Run this in a separate tab / kernel if you want to watch VRAM usage during a long experiment.

In [ ]:
# Cell 11 — VRAM Snapshot (run any time)
import torch

if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    free, total = [x / 1e9 for x in torch.cuda.mem_get_info(0)]
    print(f'Allocated : {alloc:.2f} GB')
    print(f'Reserved  : {reserved:.2f} GB')
    print(f'Free      : {free:.2f} GB  /  {total:.2f} GB total')
else:
    print('No GPU.')

## Cell 12 — Checkpoint List

List all saved AT / AT+KD checkpoints and their sizes.

In [ ]:
# Cell 12 — List Checkpoints
import os

for ckpt_dir in ['results/checkpoints/at', 'results/checkpoints/atkd']:
    print(ckpt_dir + ':')
    if not os.path.isdir(ckpt_dir):
        print('  (directory not found)')
        continue
    files = sorted(os.listdir(ckpt_dir))
    if not files:
        print('  (empty — no checkpoints yet)')
    for f in files:
        size_mb = os.path.getsize(os.path.join(ckpt_dir, f)) / 1e6
        print(f'  {f:<50}  {size_mb:6.1f} MB')
    print()